In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
from tqdm import tqdm  
import time

### dataloder设计

In [ ]:
class StockDataset_by_year(Dataset):
    def __init__(self, base_dir, start_year, end_year, task='regression'):
        self.base_dir = base_dir
        self.years = list(range(start_year, end_year + 1))
        self.task = task
        self.data = []
        self.labels = []
        
        # 加载数据并使用进度条
        for year in tqdm(self.years, desc="Loading Data by Year"):
            data_file = os.path.join(base_dir, f"{year}_df_data.feather")
            label_file = os.path.join(base_dir, f"{year}_real_RET.feather")
            
            # 读取数据和标签
            X = pd.read_feather(data_file)
            y = pd.read_feather(label_file)
            
            # 删除 'permno' 和 'DATE' 列，只保留剩余列作为特征
            X = X.drop(columns=['permno', 'DATE'])
            
            # 如果任务是分类，则将y的值转化为0或1
            if self.task == 'classification':
                y = (y > 0).astype(int)
            
            # 转换为numpy数组并存储
            self.data.append(X.values)
            self.labels.append(y.values)
        
        # 将数据和标签转换为单个数组
        self.data = torch.tensor(np.vstack(self.data), dtype=torch.float32)
        self.labels = torch.tensor(np.vstack(self.labels), dtype=torch.float32)
        
        # 打印数据集信息
        print(f"Data loaded from {start_year} to {end_year}.")
        print(f"Total samples: {len(self.labels)}")
        print(f"Feature dimension: {self.data.shape[1]}")
        print(f"Task: {self.task}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# 构建数据集和数据加载器的函数
def create_dataloaders(base_dir, start_year_train, start_year_test, end_year_test, task='regression', batch_size=32):
    # 构建训练集
    print("Constructing training dataset...")
    train_dataset = StockDataset_by_year(base_dir, start_year_train, start_year_test - 1, task)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # 构建测试集
    print("Constructing testing dataset...")
    test_dataset = StockDataset_by_year(base_dir, start_year_test, end_year_test, task)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

# 使用函数创建数据加载器
data_dir = 'data/processed/yearly'
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

In [ ]:
# 构建分类任务的数据加载器
train_loader, test_loader = create_dataloaders(data_dir, start_year_train, start_year_test, end_year_test, task='classification')

# 构建回归任务的数据加载器
train_loader_reg, test_loader_reg = create_dataloaders(data_dir, start_year_train, start_year_test, end_year_test, task='regression')

# 模型训练

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC, SVR
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor

import joblib

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import warnings
from sklearn.exceptions import DataConversionWarning
import sklearn


# 忽略DataConversionWarning
warnings.filterwarnings(action='ignore', category=DataConversionWarning)
print(sklearn.__version__)

In [ ]:
def calculate_r2_oos(real, predict):
    real = np.array(real)
    predict = np.array(predict)
    assert real.shape == predict.shape 
    valid_mask = ~np.isnan(real)
    numerator = np.sum((real[valid_mask] - predict[valid_mask]) ** 2)
    denominator = np.sum(real[valid_mask] ** 2)
    R2_oos = 1 - (numerator / denominator)
    return R2_oos

### KNN

In [ ]:
### 无进度条版本

def train_knn(train_loader, test_loader, task='classification', use_grid_search=True, save_path=None):
    # 将数据从DataLoader转换为numpy数组，并展平y
    X_train = np.vstack([x.numpy() for x, y in tqdm(train_loader, desc="Processing Training Data")])
    y_train = np.concatenate([y.numpy() for x, y in train_loader]).ravel()
    X_test = np.vstack([x.numpy() for x, y in tqdm(test_loader, desc="Processing Testing Data")])
    y_test = np.concatenate([y.numpy() for x, y in test_loader]).ravel()
    
    if task == 'classification':
        model = KNeighborsClassifier()
        param_grid = {'n_neighbors': [3, 5, 7, 9], 'weights': ['uniform', 'distance']}
    else:
        model = KNeighborsRegressor()
        param_grid = {'n_neighbors': [3, 5, 7, 9], 'weights': ['uniform', 'distance']}
    
    # 选择网格搜索或随机搜索
    if use_grid_search:
        search = GridSearchCV(model, param_grid, cv=5, n_jobs=-1, verbose=1)
    else:
        search = RandomizedSearchCV(model, param_distributions=param_grid, n_iter=10, cv=5, n_jobs=-1, verbose=1)

    # 进行搜索和训练
    search.fit(X_train, y_train)

    # 获得最佳模型
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    # 评估模型性能
    if task == 'classification':
        accuracy = accuracy_score(y_test, y_pred)
        print(f"KNN Best Accuracy: {accuracy:.4f}")
        result = {'accuracy': accuracy, 'best_params': search.best_params_}
    else:
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"KNN Best MSE: {mse:.4f}, R²: {r2:.4f}")
        result = {'mse': mse, 'r2': r2, 'best_params': search.best_params_}

    # 保存最佳模型和结果
    if save_path:
        model_save_path = os.path.join(save_path, f"KNN_{task}.joblib")
        result_save_path = os.path.join(save_path, f"KNN_{task}_results.txt")
        os.makedirs(save_path, exist_ok=True)
        joblib.dump(best_model, model_save_path)
        with open(result_save_path, 'w') as f:
            f.write(f"Best Model Parameters: {search.best_params_}\n")
            f.write(f"Validation Results: {result}\n")
        print(f"Model and results saved to {save_path}")

# 指定路径
save_dir = 'outputs/models/KNN'

# 训练和测试 KNN 回归任务
train_knn(train_loader_reg, test_loader_reg, task='regression', use_grid_search=True, save_path=save_dir)

# 训练和测试 KNN 分类任务
train_knn(train_loader, test_loader, task='classification', use_grid_search=True, save_path=save_dir)

### LR

In [ ]:
from sklearn.linear_model import LogisticRegression

def train_logistic_regression(train_loader, test_loader, use_grid_search=True, save_path=None):
    # 将数据从DataLoader转换为numpy数组，并展平y
    X_train = np.vstack([x.numpy() for x, y in tqdm(train_loader, desc="Processing Training Data")])
    y_train = np.concatenate([y.numpy() for x, y in train_loader]).ravel()
    X_test = np.vstack([x.numpy() for x, y in tqdm(test_loader, desc="Processing Testing Data")])
    y_test = np.concatenate([y.numpy() for x, y in test_loader]).ravel()
    
    model = LogisticRegression(max_iter=1000, solver='liblinear')
    param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2']}
    
    # 选择网格搜索或随机搜索
    if use_grid_search:
        search = GridSearchCV(model, param_grid, cv=5, n_jobs=2, verbose=3)  # 将 n_jobs 设置为1
    else:
        search = RandomizedSearchCV(model, param_distributions=param_grid, n_iter=3, cv=3, n_jobs=-1, verbose=3)

    # 进行搜索和训练
    print("Starting Logistic Regression model training and hyperparameter search...")
    search.fit(X_train, y_train)

    # 获得最佳模型
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    # 评估模型性能，使用分类任务的指标
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"Logistic Regression Best Accuracy: {accuracy:.4f}")
    print(f"Logistic Regression Precision: {precision:.4f}")
    print(f"Logistic Regression Recall: {recall:.4f}")
    print(f"Logistic Regression F1 Score: {f1:.4f}")
    
    result = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'best_params': search.best_params_
    }

    # 保存最佳模型和结果
    if save_path:
        model_save_path = os.path.join(save_path, "LogisticRegression_classification.joblib")
        result_save_path = os.path.join(save_path, "LogisticRegression_classification_results.txt")
        os.makedirs(save_path, exist_ok=True)
        joblib.dump(best_model, model_save_path)
        with open(result_save_path, 'w') as f:
            f.write(f"Best Model Parameters: {search.best_params_}\n")
            f.write(f"Validation Results: {result}\n")
        print(f"Model and results saved to {save_path}")

### 贝叶斯

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import joblib
import os
import warnings
from sklearn.exceptions import DataConversionWarning

# 忽略DataConversionWarning
warnings.filterwarnings(action='ignore', category=DataConversionWarning)

def train_naive_bayes(train_loader, test_loader, save_path=None):
    # 将数据从DataLoader转换为numpy数组，并展平y
    X_train = np.vstack([x.numpy() for x, y in tqdm(train_loader, desc="Processing Training Data")])
    y_train = np.concatenate([y.numpy() for x, y in train_loader]).ravel()
    X_test = np.vstack([x.numpy() for x, y in tqdm(test_loader, desc="Processing Testing Data")])
    y_test = np.concatenate([y.numpy() for x, y in test_loader]).ravel()
    
    # 创建NaiveBayes模型
    model = GaussianNB()
    print("Training Naive Bayes model...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # 评估模型性能
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"Naive Bayes Accuracy: {accuracy:.4f}")
    print(f"Naive Bayes Precision: {precision:.4f}")
    print(f"Naive Bayes Recall: {recall:.4f}")
    print(f"Naive Bayes F1 Score: {f1:.4f}")
    
    # 记录模型结果和超参数
    result = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'var_smoothing': model.var_smoothing
    }

    # 检查并创建保存路径
    if save_path:
        os.makedirs(save_path, exist_ok=True)  # 自动检查并创建文件夹
        model_save_path = os.path.join(save_path, "NaiveBayes_classification.joblib")
        result_save_path = os.path.join(save_path, "NaiveBayes_classification_results.txt")
        joblib.dump(model, model_save_path)
        with open(result_save_path, 'w') as f:
            f.write(f"Validation Results: {result}\n")
        print(f"Model and results saved to {save_path}")

# 指定路径
save_dir = 'outputs/models/NaiveBayes'

# 训练和测试 Naive Bayes 分类任务
train_naive_bayes(train_loader, test_loader, save_path=save_dir)

### 决策树

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from tqdm import tqdm
import joblib
import os
import numpy as np
import warnings
from sklearn.exceptions import DataConversionWarning

In [ ]:
def train_decision_tree(train_loader, test_loader, task='classification', n_iter=5, save_path=None):
    # 将数据从DataLoader转换为numpy数组，并展平y
    X_train = np.vstack([x.numpy() for x, y in tqdm(train_loader, desc="Processing Training Data")])
    y_train = np.concatenate([y.numpy() for x, y in train_loader]).ravel()
    X_test = np.vstack([x.numpy() for x, y in tqdm(test_loader, desc="Processing Testing Data")])
    y_test = np.concatenate([y.numpy() for x, y in test_loader]).ravel()
    
    # 根据任务类型选择模型
    if task == 'classification':
        model = DecisionTreeClassifier(random_state=42)
        param_distributions = {
            'max_depth': [None, 5, 10, 15, 20],
            'min_samples_split': [2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    else:
        model = DecisionTreeRegressor(random_state=42)
        param_distributions = {
            'max_depth': [None, 5, 10, 15, 20],
            'min_samples_split': [2, 5, 10],
            'criterion': ['squared_error', 'friedman_mse', 'absolute_error']  # 更新为支持的选项
        }
    
    # 使用随机搜索，限制迭代次数为 n_iter
    search = RandomizedSearchCV(model, param_distributions=param_distributions, n_iter=n_iter, cv=3, n_jobs=-1, verbose=1)

    # 进行搜索和训练
    print("Starting Decision Tree model training and hyperparameter search...")
    search.fit(X_train, y_train)

    # 获得最佳模型
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    # 评估模型性能
    if task == 'classification':
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        print(f"Decision Tree Best Accuracy: {accuracy:.4f}")
        print(f"Decision Tree Precision: {precision:.4f}")
        print(f"Decision Tree Recall: {recall:.4f}")
        print(f"Decision Tree F1 Score: {f1:.4f}")
        
        result = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'best_params': search.best_params_
        }
    else:
        mse = mean_squared_error(y_test, y_pred)
        r2_oos = calculate_r2_oos(y_test, y_pred)
        
        print(f"Decision Tree Best MSE: {mse:.4f}")
        print(f"Decision Tree Out-of-Sample R²: {r2_oos:.4f}")
        
        result = {
            'mse': mse,
            'r2_oos': r2_oos,
            'best_params': search.best_params_
        }

    # 检查并创建保存路径
    if save_path:
        os.makedirs(save_path, exist_ok=True)  # 自动检查并创建文件夹
        model_save_path = os.path.join(save_path, f"DecisionTree_{task}.joblib")
        result_save_path = os.path.join(save_path, f"DecisionTree_{task}_results.txt")
        joblib.dump(best_model, model_save_path)
        with open(result_save_path, 'w') as f:
            f.write(f"Best Model Parameters: {search.best_params_}\n")
            f.write(f"Validation Results: {result}\n")
        print(f"Model and results saved to {save_path}")

# 指定路径
save_dir = 'outputs/models/DecisionTree'

# 训练和测试 Decision Tree 分类任务
train_decision_tree(train_loader, test_loader, task='classification', use_grid_search=False, save_path=save_dir)

# 训练和测试 Decision Tree 回归任务
train_decision_tree(train_loader_reg, test_loader_reg, task='regression', use_grid_search=False, save_path=save_dir)

### SVM SVR

In [ ]:
import time
from sklearn.svm import SVC, SVR
from sklearn.model_selection import ParameterSampler, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from tqdm import tqdm
import joblib
import os
import numpy as np
import warnings
from sklearn.exceptions import DataConversionWarning

# 忽略DataConversionWarning
warnings.filterwarnings(action='ignore', category=DataConversionWarning)


def train_svm_svr(train_loader, test_loader, task='classification', n_iter=5, save_path=None):
    # 将数据从DataLoader转换为numpy数组，并展平y
    X_train = np.vstack([x.numpy() for x, y in tqdm(train_loader, desc="Processing Training Data")])
    y_train = np.concatenate([y.numpy() for x, y in train_loader]).ravel()
    X_test = np.vstack([x.numpy() for x, y in tqdm(test_loader, desc="Processing Testing Data")])
    y_test = np.concatenate([y.numpy() for x, y in test_loader]).ravel()
    
    # 根据任务类型选择模型
    if task == 'classification':
        model = SVC()
        param_distributions = {
            'C': [0.1, 1, 10, 100],
            'kernel': ['linear', 'poly', 'rbf'],
            'gamma': ['scale', 'auto']
        }
    else:
        model = SVR()
        param_distributions = {
            'C': [0.1, 1, 10, 100],
            'kernel': ['linear', 'poly', 'rbf'],
            'gamma': ['scale', 'auto'],
            'epsilon': [0.01, 0.1, 0.2]
        }
    
    # 使用随机搜索，限制迭代次数为 n_iter，禁用并行化（n_jobs=1）
    search = RandomizedSearchCV(model, param_distributions=param_distributions, n_iter=n_iter, cv=3, n_jobs=1, verbose=3)

    start_time = time.time()
    print("Starting SVM/SVR model training and hyperparameter search...")

    # 进行搜索和训练
    search.fit(X_train, y_train)

    elapsed_time = time.time() - start_time
    print(f"Total training time: {elapsed_time:.2f} seconds")

    # 获得最佳模型
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    # 评估模型性能
    if task == 'classification':
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        print(f"\nSVM Best Accuracy: {accuracy:.4f}")
        print(f"SVM Precision: {precision:.4f}")
        print(f"SVM Recall: {recall:.4f}")
        print(f"SVM F1 Score: {f1:.4f}")
        
        result = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'best_params': search.best_params_
        }
    else:
        mse = mean_squared_error(y_test, y_pred)
        r2_oos = calculate_r2_oos(y_test, y_pred)
        
        print(f"\nSVR Best MSE: {mse:.4f}")
        print(f"SVR Out-of-Sample R²: {r2_oos:.4f}")
        
        result = {
            'mse': mse,
            'r2_oos': r2_oos,
            'best_params': search.best_params_
        }

    # 检查并创建保存路径
    if save_path:
        os.makedirs(save_path, exist_ok=True)  # 自动检查并创建文件夹
        model_save_path = os.path.join(save_path, f"SVM_SVR_{task}.joblib")
        result_save_path = os.path.join(save_path, f"SVM_SVR_{task}_results.txt")
        joblib.dump(best_model, model_save_path)
        with open(result_save_path, 'w') as f:
            f.write(f"Best Model Parameters: {search.best_params_}\n")
            f.write(f"Validation Results: {result}\n")
        print(f"Model and results saved to {save_path}")

### adaboost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

def train_adaboost(train_loader, test_loader, task='classification', n_iter=5, save_path=None):
    # 将数据从DataLoader转换为numpy数组，并展平y
    X_train = np.vstack([x.numpy() for x, y in tqdm(train_loader, desc="Processing Training Data")])
    y_train = np.concatenate([y.numpy() for x, y in train_loader]).ravel()
    X_test = np.vstack([x.numpy() for x, y in tqdm(test_loader, desc="Processing Testing Data")])
    y_test = np.concatenate([y.numpy() for x, y in test_loader]).ravel()
    
    # 根据任务类型选择模型
    if task == 'classification':
        model = AdaBoostClassifier(estimator=DecisionTreeClassifier())
        param_distributions = {
            'n_estimators': [50, 100, 150, 200],
            'learning_rate': [0.01, 0.1, 1, 10],
            'estimator__max_depth': [1, 2, 3, 4, 5]
        }
    else:
        model = AdaBoostRegressor(estimator=DecisionTreeRegressor())
        param_distributions = {
            'n_estimators': [50, 100, 150, 200],
            'learning_rate': [0.01, 0.1, 1, 10],
            'estimator__max_depth': [1, 2, 3, 4, 5]
        }
    
    # 使用随机搜索，限制迭代次数为 n_iter，禁用并行化（n_jobs=1）
    search = RandomizedSearchCV(model, param_distributions=param_distributions, n_iter=n_iter, cv=3, n_jobs=-1, verbose=3)

    start_time = time.time()
    print("Starting AdaBoost model training and hyperparameter search...")

    # 进行搜索和训练
    search.fit(X_train, y_train)

    elapsed_time = time.time() - start_time
    print(f"Total training time: {elapsed_time:.2f} seconds")

    # 获得最佳模型
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    # 评估模型性能
    if task == 'classification':
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        print(f"\nAdaBoost Best Accuracy: {accuracy:.4f}")
        print(f"AdaBoost Precision: {precision:.4f}")
        print(f"AdaBoost Recall: {recall:.4f}")
        print(f"AdaBoost F1 Score: {f1:.4f}")
        
        result = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'best_params': search.best_params_
        }
    else:
        mse = mean_squared_error(y_test, y_pred)
        r2_oos = calculate_r2_oos(y_test, y_pred)
        
        print(f"\nAdaBoost Best MSE: {mse:.4f}")
        print(f"AdaBoost Out-of-Sample R²: {r2_oos:.4f}")
        
        result = {
            'mse': mse,
            'r2_oos': r2_oos,
            'best_params': search.best_params_
        }

    # 检查并创建保存路径
    if save_path:
        os.makedirs(save_path, exist_ok=True)  # 自动检查并创建文件夹
        model_save_path = os.path.join(save_path, f"AdaBoost_{task}.joblib")
        result_save_path = os.path.join(save_path, f"AdaBoost_{task}_results.txt")
        joblib.dump(best_model, model_save_path)
        with open(result_save_path, 'w') as f:
            f.write(f"Best Model Parameters: {search.best_params_}\n")
            f.write(f"Validation Results: {result}\n")
        print(f"Model and results saved to {save_path}")

### 随机森林

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

def train_random_forest(train_loader, test_loader, task='classification', n_iter=5, save_path=None):
    # 将数据从DataLoader转换为numpy数组，并展平y
    X_train = np.vstack([x.numpy() for x, y in tqdm(train_loader, desc="Processing Training Data")])
    y_train = np.concatenate([y.numpy() for x, y in train_loader]).ravel()
    X_test = np.vstack([x.numpy() for x, y in tqdm(test_loader, desc="Processing Testing Data")])
    y_test = np.concatenate([y.numpy() for x, y in test_loader]).ravel()
    
    # 根据任务类型选择模型
    if task == 'classification':
        model = RandomForestClassifier(random_state=42)
        param_distributions = {
            'n_estimators': [50, 100, 200, 300],
            'max_depth': [None, 10, 20, 30, 40],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'bootstrap': [True, False]
        }
    else:
        model = RandomForestRegressor(random_state=42)
        param_distributions = {
            'n_estimators': [50, 100, 200, 300],
            'max_depth': [None, 10, 20, 30, 40],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'bootstrap': [True, False]
        }
    
    # 使用随机搜索，限制迭代次数为 n_iter，禁用并行化（n_jobs=1）
    search = RandomizedSearchCV(model, param_distributions=param_distributions, n_iter=n_iter, cv=3, n_jobs=-1, verbose=3)

    start_time = time.time()
    print("Starting Random Forest model training and hyperparameter search...")

    # 进行搜索和训练
    search.fit(X_train, y_train)

    elapsed_time = time.time() - start_time
    print(f"Total training time: {elapsed_time:.2f} seconds")

    # 获得最佳模型
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    # 评估模型性能
    if task == 'classification':
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        print(f"\nRandom Forest Best Accuracy: {accuracy:.4f}")
        print(f"Random Forest Precision: {precision:.4f}")
        print(f"Random Forest Recall: {recall:.4f}")
        print(f"Random Forest F1 Score: {f1:.4f}")
        
        result = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'best_params': search.best_params_
        }
    else:
        mse = mean_squared_error(y_test, y_pred)
        r2_oos = calculate_r2_oos(y_test, y_pred)
        
        print(f"\nRandom Forest Best MSE: {mse:.4f}")
        print(f"Random Forest Out-of-Sample R²: {r2_oos:.4f}")
        
        result = {
            'mse': mse,
            'r2_oos': r2_oos,
            'best_params': search.best_params_
        }

    # 检查并创建保存路径
    if save_path:
        os.makedirs(save_path, exist_ok=True)  # 自动检查并创建文件夹
        model_save_path = os.path.join(save_path, f"RandomForest_{task}.joblib")
        result_save_path = os.path.join(save_path, f"RandomForest_{task}_results.txt")
        joblib.dump(best_model, model_save_path)
        with open(result_save_path, 'w') as f:
            f.write(f"Best Model Parameters: {search.best_params_}\n")
            f.write(f"Validation Results: {result}\n")
        print(f"Model and results saved to {save_path}")

### XGBOOST

In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV

def train_xgboost(train_loader, test_loader, task='classification', n_iter=5, save_path=None):
    # 将数据从DataLoader转换为numpy数组，并展平y
    X_train = np.vstack([x.numpy() for x, y in tqdm(train_loader, desc="Processing Training Data")])
    y_train = np.concatenate([y.numpy() for x, y in train_loader]).ravel()
    X_test = np.vstack([x.numpy() for x, y in tqdm(test_loader, desc="Processing Testing Data")])
    y_test = np.concatenate([y.numpy() for x, y in test_loader]).ravel()
    
    # 根据任务类型选择模型
    if task == 'classification':
        model = xgb.XGBClassifier(objective='binary:logistic', use_label_encoder=False, eval_metric='logloss', random_state=42)
        param_distributions = {
            'n_estimators': [50, 100, 200, 300],
            'max_depth': [3, 6, 10],
            'learning_rate': [0.01, 0.1, 0.2, 0.3],
            'subsample': [0.5, 0.7, 1.0],
            'colsample_bytree': [0.5, 0.7, 1.0]
        }
    else:
        model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
        param_distributions = {
            'n_estimators': [50, 100, 200, 300],
            'max_depth': [3, 6, 10],
            'learning_rate': [0.01, 0.1, 0.2, 0.3],
            'subsample': [0.5, 0.7, 1.0],
            'colsample_bytree': [0.5, 0.7, 1.0]
        }
    
    # 使用随机搜索，限制迭代次数为 n_iter，禁用并行化（n_jobs=1）
    search = RandomizedSearchCV(model, param_distributions=param_distributions, n_iter=n_iter, cv=3, n_jobs=-1, verbose=3)

    start_time = time.time()
    print("Starting XGBoost model training and hyperparameter search...")

    # 进行搜索和训练
    search.fit(X_train, y_train)

    elapsed_time = time.time() - start_time
    print(f"Total training time: {elapsed_time:.2f} seconds")

    # 获得最佳模型
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    # 评估模型性能
    if task == 'classification':
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        print(f"\nXGBoost Best Accuracy: {accuracy:.4f}")
        print(f"XGBoost Precision: {precision:.4f}")
        print(f"XGBoost Recall: {recall:.4f}")
        print(f"XGBoost F1 Score: {f1:.4f}")
        
        result = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'best_params': search.best_params_
        }
    else:
        mse = mean_squared_error(y_test, y_pred)
        r2_oos = calculate_r2_oos(y_test, y_pred)
        
        print(f"\nXGBoost Best MSE: {mse:.4f}")
        print(f"XGBoost Out-of-Sample R²: {r2_oos:.4f}")
        
        result = {
            'mse': mse,
            'r2_oos': r2_oos,
            'best_params': search.best_params_
        }

    # 检查并创建保存路径
    if save_path:
        os.makedirs(save_path, exist_ok=True)  # 自动检查并创建文件夹
        model_save_path = os.path.join(save_path, f"XGBoost_{task}.joblib")
        result_save_path = os.path.join(save_path, f"XGBoost_{task}_results.txt")
        joblib.dump(best_model, model_save_path)
        with open(result_save_path, 'w') as f:
            f.write(f"Best Model Parameters: {search.best_params_}\n")
            f.write(f"Validation Results: {result}\n")
        print(f"Model and results saved to {save_path}")

## 单独SVR

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch
from tqdm import tqdm

# SVR模型训练函数
def train_svr(train_loader, test_loader, kernel='rbf', C=1.0, epsilon=0.1):
    # 初始化SVR模型
    svr = SVR(kernel=kernel, C=C, epsilon=epsilon)
    
    # 用于标准化数据的scaler
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    # 收集训练集数据并进行标准化
    X_train, y_train = [], []
    for X_batch, y_batch in tqdm(train_loader, desc="Processing Training Data"):
        X_train.append(X_batch.numpy())
        y_train.append(y_batch.numpy())
    X_train = np.vstack(X_train)
    y_train = np.vstack(y_train).ravel()
    
    X_train = scaler_X.fit_transform(X_train)
    y_train = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
    
    # 训练SVR模型
    print("Training SVR model...")
    svr.fit(X_train, y_train)
    print("SVR model training completed.")
    
    # 收集测试集数据并进行标准化
    X_test, y_test = [], []
    for X_batch, y_batch in tqdm(test_loader, desc="Processing Testing Data"):
        X_test.append(X_batch.numpy())
        y_test.append(y_batch.numpy())
    X_test = np.vstack(X_test)
    y_test = np.vstack(y_test).ravel()
    
    X_test = scaler_X.transform(X_test)
    y_test = scaler_y.transform(y_test.reshape(-1, 1)).ravel()
    
    # 模型预测
    print("Predicting with SVR model...")
    y_pred = svr.predict(X_test)
    
    # 反标准化预测结果
    y_pred_rescaled = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).ravel()
    y_test_rescaled = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()
    
    # 计算MSE和R^2
    mse = mean_squared_error(y_test_rescaled, y_pred_rescaled)
    r2 = r2_score(y_test_rescaled, y_pred_rescaled)
    r2oos = calculate_r2_oos(y_test_rescaled, y_pred_rescaled)
    
    print(f"Mean Squared Error: {mse}")
    print(f"R^2 Score: {r2}")
    print(f"R^2oos Score: {r2oos}")
    
    return svr

# 使用数据加载器训练SVR模型
svr_model = train_svr(train_loader_reg, test_loader_reg, kernel='rbf', C=10.0, epsilon=0.05)